<a href="https://colab.research.google.com/github/Pumlisa97/mit805-airline-delays-pyspark/blob/main/notebooks/02_part1_pyspark_preparation_and_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: PySpark Data Preparation and Exploratory Data Analysis

## MIT 805 Big Data Semester Project

**Dataset:** U.S. Department of Transportation Bureau of Transportation Statistics
(BTS) Reporting Carrier On-Time Performance (1987–present)

**Working dataset:** January 2020–December 2024, 60 monthly CSV files  
**Raw public dataset:** January 2015–December 2024, 120 monthly CSV files  
**Research question:** How do departure delays, arrival delays, and cancellations
vary by airline, origin airport, month, and scheduled departure period, and which
operational delay categories account for the greatest disruption?

This notebook prepares the PySpark processing dataset and performs initial data
quality assessment and exploratory data analysis. Large-scale transformations and
aggregations are performed in PySpark. Pandas is used only for small result tables
or visualisations after aggregation.

In [1]:
!pip -q install pyspark==3.5.1 pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/MIT805_Airline_Delays/data")
WORKING_DIR = DATA_ROOT / "working"
PROCESSED_DIR = DATA_ROOT / "processed"
TEMP_DIR = DATA_ROOT / "temp"

print(f"Working directory exists: {WORKING_DIR.exists()}")
print(f"Processed directory exists: {PROCESSED_DIR.exists()}")
print(f"Working CSV files: {len(list(WORKING_DIR.glob('*.csv')))}")

Mounted at /content/drive
Working directory exists: True
Processed directory exists: True
Working CSV files: 60


In [3]:
import pyspark

print(f"PySpark version: {pyspark.__version__}")

PySpark version: 3.5.1


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MIT805_Airline_Delays_Part1")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

Spark version: 3.5.1
Application name: MIT805_Airline_Delays_Part1
Default parallelism: 2


In [5]:
from pathlib import Path

single_files = sorted(WORKING_DIR.glob("*2020_1.csv"))

if not single_files:
    raise FileNotFoundError(
        "Could not find the January 2020 CSV. "
        "Check WORKING_DIR and list its contents."
    )

single_file = single_files[0]

print(f"Testing file: {single_file.name}")

test_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(str(single_file))
)

print(f"Columns found: {len(test_df.columns)}")
print(f"Last five columns: {test_df.columns[-5:]}")

test_df.select(
    "Year",
    "Month",
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "DepDelay",
    "ArrDelay",
    "Cancelled"
).show(5, truncate=False)

Testing file: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_1.csv
Columns found: 110
Last five columns: ['Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum', '_c109']
+----+-----+----------+-----------------+------+----+--------+--------+---------+
|Year|Month|FlightDate|Reporting_Airline|Origin|Dest|DepDelay|ArrDelay|Cancelled|
+----+-----+----------+-----------------+------+----+--------+--------+---------+
|2020|1    |2020-01-17|B6               |PBI   |BDL |-14.00  |-11.00  |0.00     |
|2020|1    |2020-01-18|B6               |PBI   |BDL |-14.00  |-9.00   |0.00     |
|2020|1    |2020-01-19|B6               |PBI   |BDL |7.00    |-6.00   |0.00     |
|2020|1    |2020-01-20|B6               |PBI   |BDL |-7.00   |4.00    |0.00     |
|2020|1    |2020-01-21|B6               |PBI   |BDL |12.00   |6.00    |0.00     |
+----+-----+----------+-----------------+------+----+--------+--------+---------+
only showing top 5 rows



In [6]:
from pyspark.sql import functions as F

working_pattern = str(WORKING_DIR / "*.csv")

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(working_pattern)
    .withColumn("source_file", F.input_file_name())
)

print(f"Columns read: {len(raw_df.columns)}")
print(f"Input partitions: {raw_df.rdd.getNumPartitions()}")

Columns read: 111
Input partitions: 112


In [7]:
from pyspark.sql.types import (
    IntegerType,
    DoubleType,
    DateType,
)

core_string_columns = [
    "Reporting_Airline",
    "IATA_CODE_Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "OriginCityName",
    "OriginState",
    "OriginStateName",
    "Dest",
    "DestCityName",
    "DestState",
    "DestStateName",
    "DepTimeBlk",
    "ArrTimeBlk",
    "CancellationCode",
]

integer_columns = [
    "Year",
    "Quarter",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "DOT_ID_Reporting_Airline",
    "CRSDepTime",
    "DepTime",
    "DepDel15",
    "CRSArrTime",
    "ArrTime",
    "ArrDel15",
    "Cancelled",
    "Diverted",
    "DistanceGroup",
]

double_columns = [
    "DepDelay",
    "DepDelayMinutes",
    "ArrDelay",
    "ArrDelayMinutes",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
]

selected_df = raw_df.select(
    *[F.col(column).cast("string").alias(column) for column in core_string_columns],
    *[F.col(column).cast("int").alias(column) for column in integer_columns],
    *[F.to_date(F.col("FlightDate"), "yyyy-MM-dd").alias("FlightDate")],
    *[F.col(column).cast("double").alias(column) for column in double_columns],
    F.col("source_file")
)

clean_df = (
    selected_df
    .filter(F.col("Year").between(2020, 2024))
    .filter(F.col("Month").between(1, 12))
    .filter(F.col("FlightDate").isNotNull())
    .withColumn("ScheduledDepHour", F.floor(F.col("CRSDepTime") / 100).cast("int"))
    .withColumn("FlightMonth", F.trunc(F.col("FlightDate"), "month"))
)

print(f"Core columns retained: {len(selected_df.columns) - 1}")
print(f"Columns after derived fields: {len(clean_df.columns)}")

clean_df.printSchema()

Core columns retained: 43
Columns after derived fields: 46
root
 |-- Reporting_Airline: string (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- DestStateName: string (nullable = true)
 |-- DepTimeBlk: string (nullable = true)
 |-- ArrTimeBlk: string (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- CRSDepTime: integer (nullable =

In [10]:
import shutil

PROCESSING_PATH = PROCESSED_DIR / "airline_core_2020_2024"

if PROCESSING_PATH.exists():
    shutil.rmtree(PROCESSING_PATH)
    print("Removed previous processing output.")

(
    clean_df
    .repartition(60, "Year", "Month")
    .write
    .mode("overwrite")
    .partitionBy("Year", "Month")
    .parquet(str(PROCESSING_PATH))
)

print(f"Processing dataset written to: {PROCESSING_PATH}")

Processing dataset written to: /content/drive/MyDrive/MIT805_Airline_Delays/data/processed/airline_core_2020_2024


In [11]:
def folder_size_bytes(folder: Path) -> int:
    return sum(
        path.stat().st_size
        for path in folder.rglob("*")
        if path.is_file()
    )

processing_bytes = folder_size_bytes(PROCESSING_PATH)

print(f"Processing dataset size: {processing_bytes / (1024 ** 3):.3f} GiB")
print(f"Processing dataset size: {processing_bytes / 1_000_000_000:.3f} GB")

parquet_files = list(PROCESSING_PATH.rglob("*.parquet"))
print(f"Parquet data files: {len(parquet_files)}")

partition_dirs = [
    path for path in PROCESSING_PATH.rglob("Month=*")
    if path.is_dir()
]
print(f"Year-month partitions: {len(partition_dirs)}")

Processing dataset size: 0.631 GiB
Processing dataset size: 0.677 GB
Parquet data files: 60
Year-month partitions: 60


In [12]:
FULL_PROCESSING_PATH = PROCESSED_DIR / "airline_full_2020_2024_uncompressed"

if FULL_PROCESSING_PATH.exists():
    shutil.rmtree(FULL_PROCESSING_PATH)
    print("Removed previous full-schema processing output.")

full_processing_df = (
    raw_df
    .filter(F.col("Year").cast("int").between(2020, 2024))
    .filter(F.col("Month").cast("int").between(1, 12))
    .drop("_c109")
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("Month", F.col("Month").cast("int"))
    .withColumn("FlightDate", F.to_date(F.col("FlightDate"), "yyyy-MM-dd"))
    .filter(F.col("FlightDate").isNotNull())
)

(
    full_processing_df
    .repartition(60, "Year", "Month")
    .write
    .mode("overwrite")
    .option("compression", "uncompressed")
    .partitionBy("Year", "Month")
    .parquet(str(FULL_PROCESSING_PATH))
)

print(f"Full processing dataset written to: {FULL_PROCESSING_PATH}")

Full processing dataset written to: /content/drive/MyDrive/MIT805_Airline_Delays/data/processed/airline_full_2020_2024_uncompressed


In [13]:
full_processing_bytes = folder_size_bytes(FULL_PROCESSING_PATH)

print(f"Full processing dataset size: {full_processing_bytes / (1024 ** 3):.3f} GiB")
print(f"Full processing dataset size: {full_processing_bytes / 1_000_000_000:.3f} GB")

full_parquet_files = list(FULL_PROCESSING_PATH.rglob("*.parquet"))
print(f"Full Parquet data files: {len(full_parquet_files)}")

Full processing dataset size: 1.125 GiB
Full processing dataset size: 1.208 GB
Full Parquet data files: 60


In [14]:
CSV_PROCESSING_PATH = PROCESSED_DIR / "airline_full_2020_2024_csv"

if CSV_PROCESSING_PATH.exists():
    shutil.rmtree(CSV_PROCESSING_PATH)
    print("Removed previous CSV processing output.")

(
    full_processing_df
    .repartition(60, "Year", "Month")
    .write
    .mode("overwrite")
    .option("header", True)
    .option("compression", "none")
    .partitionBy("Year", "Month")
    .csv(str(CSV_PROCESSING_PATH))
)

print(f"CSV processing dataset written to: {CSV_PROCESSING_PATH}")

CSV processing dataset written to: /content/drive/MyDrive/MIT805_Airline_Delays/data/processed/airline_full_2020_2024_csv


In [15]:
csv_processing_bytes = folder_size_bytes(CSV_PROCESSING_PATH)

print(f"CSV processing dataset size: {csv_processing_bytes / (1024 ** 3):.3f} GiB")
print(f"CSV processing dataset size: {csv_processing_bytes / 1_000_000_000:.3f} GB")

csv_part_files = list(CSV_PROCESSING_PATH.rglob("*.csv"))
print(f"CSV processing data files: {len(csv_part_files)}")

csv_partition_dirs = [
    path for path in CSV_PROCESSING_PATH.rglob("Month=*")
    if path.is_dir()
]
print(f"Year-month partitions: {len(csv_partition_dirs)}")

CSV processing dataset size: 14.586 GiB
CSV processing dataset size: 15.662 GB
CSV processing data files: 60
Year-month partitions: 60
